In [9]:
import os
from dotenv import load_dotenv

load_dotenv()

pinecone_key = os.getenv("PINECONE_API_KEY")
openai_key = os.getenv("OPENAI_API_KEY")

In [10]:
from pinecone import Pinecone, ServerlessSpec

pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])
index_name = "langchain-demo"

if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=1536,          # must match your embedding model
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )

index = pc.Index(index_name)

In [7]:
from langchain_openai import OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")  # 1536 dims
vector_store = PineconeVectorStore(index=index, embedding=embeddings)

ModuleNotFoundError: No module named 'langchain_pinecone'

In [4]:
from langchain_core.documents import Document

docs = [
    Document(page_content="Pinecone is a managed vector database.", metadata={"source": "notes"}),
    Document(page_content="LangChain helps build LLM applications.", metadata={"source": "notes"}),
]
vector_store.add_documents(docs, ids=["doc1", "doc2"])

['doc1', 'doc2']

In [5]:
results = vector_store.similarity_search("What is Pinecone?", k=2)
for r in results:
    print(r.page_content, r.metadata)

# With scores and a metadata filter
results = vector_store.similarity_search_with_score(
    "vector database", k=2, filter={"source": "notes"}
)

Pinecone is a managed vector database. {'source': 'notes'}
LangChain helps build LLM applications. {'source': 'notes'}


In [6]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 3})
context_docs = retriever.invoke("Explain Pinecone")

In [7]:
context_docs

[Document(id='doc1', metadata={'source': 'notes'}, page_content='Pinecone is a managed vector database.'),
 Document(id='doc2', metadata={'source': 'notes'}, page_content='LangChain helps build LLM applications.')]

In [8]:
from langchain_core.documents import Document

new_docs = [
    Document(page_content="Agentic AI systems plan and use tools.", metadata={"topic": "agents"}),
    Document(page_content="RAG combines retrieval with generation.", metadata={"topic": "rag"}),
]

vector_store.add_documents(new_docs, ids=["doc3", "doc4"])

['doc3', 'doc4']